# 🛡️ अल्टीमेट क्लाउड एंटीवायरस और प्राइवेसी सैंडबॉक्स

In [ ]:
import os, sys, shutil, glob
from IPython.display import FileLink, display

# ------------------------------------------------------------
# 0. Sabhi zaroori tools install karo
# ------------------------------------------------------------
print('🔧 Tools install ho rahe hain...')
!apt-get update -y
!apt-get install -y clamav clamav-daemon unzip unrar p7zip-full wget file
!freshclam
print('✅ Sab tools tayar hain.\n')

print('🔧 mediafire-dl install kar rahe hain...')
!pip install -q git+https://github.com/Juvenal-Yescas/mediafire-dl.git
print('✅ mediafire-dl ready.\n')

# ------------------------------------------------------------
# 1. Link input
# ------------------------------------------------------------
print('🔒 Privacy active.')
mf_link = input('➡️ MediaFire ya file link paste karo aur Enter dabao: ')

# ------------------------------------------------------------
# 2. Download
# ------------------------------------------------------------
print('\n⏳ Downloading...')
# Purane archives saaf karo (sirf .zip, .rar, .7z)
for f in glob.glob('*.zip') + glob.glob('*.rar') + glob.glob('*.7z'):
    try:
        os.remove(f)
    except:
        pass
os.system(f'mediafire-dl "{mf_link}"')

# ------------------------------------------------------------
# 3. Downloaded archive dhundho (hidden files ignore)
# ------------------------------------------------------------
# Sirf .zip, .rar, .7z extensions wali files dhundho
archive_files = glob.glob('*.zip') + glob.glob('*.rar') + glob.glob('*.7z')
# Hidden files (jo '.' se start ho) hata do
archive_files = [f for f in archive_files if not os.path.basename(f).startswith('.')]

if not archive_files:
    print('❌ Koi archive file (.zip, .rar, .7z) download nahi hui.')
    print('   Ho sakta hai link galat ho ya file direct download na ho.')
    raise SystemExit(1)  # cell yahin ruk jayega, aage nahi badhega

# Sabse nayi archive file pick karo (safety ke liye, agar multiple ho)
archive_file = max(archive_files, key=lambda f: os.path.getmtime(f))
print(f'📥 Downloaded archive: {archive_file}')
print(f'📏 Size: {os.path.getsize(archive_file)} bytes')

# --- File type analysis ---
print('\n🔍 File type:')
file_output = !file "{archive_file}"
print(file_output[0])

# --- Archive ke andar ki file list dikhao ---
ext = archive_file.split('.')[-1].lower()
print(f'\n📂 Contents of {archive_file}:')
if ext == 'zip':
    !unzip -l "{archive_file}"
elif ext == 'rar':
    !unrar l "{archive_file}"
elif ext == '7z':
    !7z l "{archive_file}"
else:
    print('(Unknown format, phir bhi extract try karenge)')

# ------------------------------------------------------------
# 4. Extract
# ------------------------------------------------------------
print('\n⏳ Extracting...')
if os.path.exists('extracted_files'):
    shutil.rmtree('extracted_files')
os.makedirs('extracted_files', exist_ok=True)

if ext == 'zip':
    !unzip -o "{archive_file}" -d extracted_files/
elif ext == 'rar':
    !unrar x -o+ "{archive_file}" extracted_files/
elif ext == '7z':
    !7z x "{archive_file}" -oextracted_files/ -aoa
else:
    # Fallback: unzip se try karo
    !unzip -o "{archive_file}" -d extracted_files/ 2>/dev/null || echo 'Extraction fail.'
    if not os.listdir('extracted_files'):
        print('❌ Extraction fail. File corrupt ya unsupported format hai.')
        raise SystemExit(1)
print('✅ Extraction complete.\n')

# ------------------------------------------------------------
# 5. Scan with ClamAV
# ------------------------------------------------------------
print('🛡️ Scanning...')
!clamscan --version
scan_result = !clamscan -r --remove extracted_files/

virus_found = False
for line in scan_result:
    if 'Infected files:' in line:
        print(line.strip())
        if 'Infected files: 0' not in line:
            virus_found = True

print('\n--- Scan Summary ---')
for line in scan_result[-10:]:
    print(line)

# ------------------------------------------------------------
# 6. Result & Download
# ------------------------------------------------------------
print('\n' + '='*50)
if virus_found:
    print('❌ Virus mila! File server se delete kar di gayi.')
    print('🛑 Download blocked!')
else:
    print('✅ File 100% safe hai. Clean zip bana rahe hain...')
    shutil.make_archive('safe_download', 'zip', 'extracted_files')
    print('📥 Safe download ready:')
    display(FileLink('safe_download.zip'))
print('='*50)